# Roof Edge Tracing on Undistorted Panoramic Frames

Traces the visible roof edges of one building:

1. query pano metadata near the building and, for every pano, pick in code the ground camera
   and view that show the whole roof (about `ROOF_WIDTH_M` wide, eave to ridge) without black
   border; rank the views by viewing angle and distance, one per pano, spread across drives;
2. render each view with the calibrated rosette lens model, so straight roof lines are
   straight, and skip views where foliage hides the roof;
3. ask Gemini for `RoofEdges`: typed polylines (ridge, eave, hip, valley, rake) on a 0-1000
   grid;
4. validate every polyline against the image in code: an edge is kept only where the image has
   an oriented gradient and a straight line segment along it, away from foliage, sky and
   no-data pixels. Kept edges are snapped onto the image lines; rejected edges are drawn dashed
   in grey. Each panel reports how often random segments of the same image would pass (the
   false-accept baseline).

**Data model.** Each pano is a rosette of 7 frames (`observation_id` suffix `_0`..`_6`):
cameras 0-5 look horizontally ~60 deg apart, camera 6 looks at the sky. Frames are portrait
3648x5472 wide-angle images with a per-camera `camera_pose` (position, heading, pitch, roll)
and no published intrinsics; `svi_geo` ships a calibrated fisheye model for them. Panos are
chained into drive sequences from `snapshot_id` + capture time + distance, which is used here
to spread the chosen views over different drives.

**Division of labour.** All geometry is code: camera choice, view framing and rendering with
the calibrated lens model, and edge validation and snapping. Gemini only proposes roof
polylines on views that code has rendered; every reply is validated against a Pydantic schema
and range-checked.

**Terms of use.** The frames come from your Imagery Insights dataset and are governed by the
Imagery Insights terms of your Google Cloud agreement.

- Do not redistribute the frames, or crops rendered from them, outside your organisation.
- Frames are downloaded with your own credentials. They are not cached on disk unless you pass
  `cache_dir=` to `GcsImageFetcher`. Cached frames expire after 24 hours
  (`images.DEFAULT_CACHE_TTL_S`), and `fetcher.purge_expired()` deletes stale files.
- Keep the attribution: every figure that shows imagery is labelled "Imagery © Google", and
  every map carries the same credit next to the base-map attribution.

In [ ]:
# Install the shared `svi_geo` helper package (geometry, sequencing, dedup, Gemini client).
# In a clone of the repo the local copy is used: $SVI_GEO_LOCAL_PATH, or the first parent
# directory that contains street_view_insights/panoramic/svi_geo. Otherwise pip installs it
# from GitHub at SVI_GEO_REF.
import importlib.util
import os
import pathlib
import subprocess
import sys

# TODO(after-merge): set to "main" once svi_geo is merged. Override with $SVI_GEO_REF
# (a commit SHA gives reproducible runs).
SVI_GEO_REF = os.environ.get("SVI_GEO_REF", "svi-gemini-improvements")


def find_local_svi_geo():
    if os.environ.get("SVI_GEO_LOCAL_PATH"):
        return os.environ["SVI_GEO_LOCAL_PATH"]
    here = pathlib.Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "street_view_insights/panoramic/svi_geo" / "pyproject.toml").is_file():
            return str(d / "street_view_insights/panoramic/svi_geo")
    return None


LOCAL_SVI_GEO = find_local_svi_geo()
if importlib.util.find_spec("svi_geo") is None:
    if LOCAL_SVI_GEO:
        src = f"{LOCAL_SVI_GEO}[notebooks]"
    else:
        src = (
            "svi-geo[notebooks] @ git+https://github.com/googlemaps-samples/insights-samples"
            f"@{SVI_GEO_REF}#subdirectory=street_view_insights/panoramic/svi_geo"
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", src])

**Authentication and permissions.**

- **Colab:** the next cell calls `google.colab.auth.authenticate_user()`.
- **Local Jupyter:** Application Default Credentials, set up once with
  `gcloud auth application-default login`.
- **Optional workstation helpers** (only if plain ADC fails, e.g. Vertex AI returns 401 on
  some managed workstations): `SVI_USE_GCLOUD_TOKEN=1` mints short-lived user tokens with
  `gcloud auth print-access-token`, and `SVI_ECP_PROXY_URL=http://localhost:<port>` routes
  Vertex AI calls through a local enterprise-certificate (mTLS) proxy.
- **Parameters:** set `PROJECT_ID` (your billing project) and `GCS_BUCKET` (the frame bucket
  linked to your Imagery Insights dataset) in the parameters cell or as environment
  variables. The notebook stops if either is missing; nothing is guessed.
- **IAM roles you need** (this notebook changes no IAM settings): `roles/bigquery.jobUser` on
  `PROJECT_ID`, `roles/bigquery.dataViewer` on the linked dataset,
  `roles/storage.objectViewer` on the frame bucket and `roles/aiplatform.user` for Gemini.

In [ ]:
# Colab: authenticate the notebook user (Application Default Credentials elsewhere).
try:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
except ImportError:
    pass

In [ ]:
import os

# ---- Parameters -------------------------------------------------------------------------
PROJECT_ID = os.environ.get("PROJECT_ID", "YOUR_PROJECT_ID")  # billing project (required)
DATASET_ID = "imagery_insights___us"  # Imagery Insights dataset linked into PROJECT_ID
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")  # frame bucket linked to DATASET_ID (required)
MODEL = "gemini-3.5-flash"
LOCATION = "global"
MAX_GEMINI_CALLS = 500  # hard cap on Gemini requests for this notebook; None = unlimited
CONCURRENCY = 16  # parallel Gemini requests

LAT, LNG = 28.05047, -81.96015  # target building
RADIUS_M = 60.0
N_VIEWS = 3  # views to trace (one Gemini call each)
VIEW_W, VIEW_H = 1200, 900

In [ ]:
from google.cloud import storage

from svi_geo import attribution, auth, config, data, images, rosette
from svi_geo import gemini_client as gc

# Fails fast (ConfigError) if PROJECT_ID or GCS_BUCKET is missing; no silent fallbacks.
settings = config.resolve_settings(PROJECT_ID, GCS_BUCKET, env=dict(os.environ), dataset=DATASET_ID)
PROJECT_ID, BUCKET = settings.project, settings.bucket
print("project:", PROJECT_ID, "| frames bucket:", BUCKET)

creds = auth.get_credentials()  # None -> Application Default Credentials
TABLE = data.pano_table(PROJECT_ID, DATASET_ID)  # pano_observations_latest
bq = data.QueryRunner(
    data.make_bigquery_client(PROJECT_ID, creds),
    allowed_tables=data.pano_tables(PROJECT_ID, DATASET_ID),  # pano views only
    cache_dir=data.DEFAULT_QUERY_CACHE,
)
fetcher = images.GcsImageFetcher(storage.Client(project=PROJECT_ID, credentials=creds))
runner = gc.GeminiRunner(
    gc.VertexGeminiBackend(gc.make_vertex_client(PROJECT_ID, LOCATION, creds), model=MODEL),
    max_calls=MAX_GEMINI_CALLS,
    concurrency=CONCURRENCY,
)
intr = rosette.load_intrinsics()  # self-calibrated 7-camera rosette model shipped with svi_geo
print("intrinsics fitted:", intr.fitted)

In [ ]:
import numpy as np

from svi_geo import roof, schemas, sequence, views

In [ ]:
# Metadata only (no gcs_uri column: selecting it scans ~1.9 GB); the frame URI is derived
# from snapshot_id + observation_id. Parameterised SQL, dry-run first, 2 GB cap.
PANO_META_SQL = data.pano_meta_sql(TABLE)
params = data.pano_meta_params(lat=LAT, lng=LNG, radius_m=RADIUS_M)
print(f"dry run: {bq.dry_run(PANO_META_SQL, params) / 1e9:.2f} GB")
frames = data.normalize_frames(bq.run(PANO_META_SQL, params))
frames["gcs_uri"] = [
    data.gcs_uri_for(BUCKET, s, o)
    for s, o in zip(frames["snapshot_id"], frames["observation_id"], strict=True)
]
panos = sequence.build_sequences(data.panos_from_frames(frames))
frames = frames.merge(panos[["pano_id", "seq_id"]], on="pano_id", how="left")
print(f"{len(frames)} frames, {len(panos)} panos, {panos.seq_id.nunique()} drive sequences")

## Choose the views in code

In [ ]:
roof_views = views.rank_roof_views(
    frames, LAT, LNG, intr, n=2 * N_VIEWS, aspect=VIEW_W / VIEW_H, max_dist_m=RADIUS_M
)
if roof_views.empty:
    raise RuntimeError(
        "No pano within RADIUS_M has a camera that shows the whole roof without black border. "
        "Move LAT/LNG onto the building or increase RADIUS_M."
    )
roof_views[["pano_id", "seq_id", "cam_k", "dist_m", "off_axis_deg", "hfov", "pitch", "score"]]

In [ ]:
def render(row):
    img = images.decode(fetcher.fetch(row["gcs_uri"]))
    return roof.render_roof_view(
        img,
        intr,
        row["camera_pose"],
        float(row["bearing"]),
        int(row["cam_k"]),
        pitch_deg=float(row["pitch"]),
        hfov_deg=float(row["hfov"]),
        width=VIEW_W,
        height=VIEW_H,
    )


def roof_box(row, view):
    """Pixel box around the target roof: ROOF_WIDTH_M wide, eave to ridge, in the view."""
    u, _, _ = view.bearing_to_pixel(row["bearing"], 0.0)
    half_w = (
        VIEW_W / 2 * views.ROOF_WIDTH_M / 2 / row["dist_m"] / np.tan(np.radians(view.hfov_deg / 2))
    )
    el_eave = np.degrees(np.arctan((views.ROOF_EAVE_M - views.CAM_HEIGHT_M) / row["dist_m"]))
    el_top = np.degrees(np.arctan((views.ROOF_TOP_M - views.CAM_HEIGHT_M) / row["dist_m"]))
    _, v_eave, _ = view.bearing_to_pixel(row["bearing"], el_eave)
    _, v_top, _ = view.bearing_to_pixel(row["bearing"], el_top)
    return (float(u) - half_w, float(v_top), float(u) + half_w, float(v_eave))


chosen = []
for row in roof_views.to_dict("records"):
    img, view, black = render(row)
    screen = views.occlusion_screen(img, roof_box(row, view))
    print(
        f"{row['pano_id'][:10]} cam {row['cam_k']} {row['dist_m']:.0f} m: "
        f"black {black:.4f}, foliage {screen['foliage_frac']:.2f}"
        + (" -> skipped (foliage)" if screen["rejected"] else "")
    )
    if not screen["rejected"]:
        chosen.append({**row, "image": img, "view": view, "black": black})
    if len(chosen) == N_VIEWS:
        break
if not chosen:
    raise RuntimeError("Every candidate view of the roof is hidden by foliage.")
print(f"black_fraction_max={max(c['black'] for c in chosen):.4f}")
assert max(c["black"] for c in chosen) < 0.01, "a rendered view extends past the sensor"

In [ ]:
PROMPT = (
    "This is a lens-corrected street-level photo. Trace the visible roof edges of the main "
    "building (ridge, eaves, hips, valleys, rakes) as polylines of [y, x] points on a 0-1000 "
    "grid. Only include edges you can see; set roof_visible=false if no roof is visible."
)
# The API enforces the RoofEdges schema (JSON mode); code then range-checks every polyline
# (>= 2 distinct points, all in 0-1000) and re-asks once if a reply fails.
print(f"planned: {len(chosen)} calls, est. ${gc.estimate_cost(len(chosen)):.3f}")
roofs = await runner.ask_many(
    [([PROMPT, c["image"]], schemas.RoofEdges) for c in chosen],
    validator=lambda r: r.validate_in_image(VIEW_W, VIEW_H),
)
print(runner.cost.summary())
runner.check()  # raises if any request failed, with the first error messages
assert any(r is not None and r.roof_visible for r in roofs), "Gemini traced no roof in any view"

In [ ]:
def to_pixels(edge):
    return edge.edge_type.value, [(x / 1000 * VIEW_W, y / 1000 * VIEW_H) for y, x in edge.points]


results = []
for c, rf in zip(chosen, roofs, strict=True):
    proposed = [to_pixels(e) for e in (rf.edges if rf else [])]
    res = roof.validate_roof_edges(
        c["image"],
        proposed,
        valid_mask=c["image"].max(axis=2) > 0,
        horizon_row=roof.horizon_row_for(c["view"]),
    )
    results.append(res)
    print(
        f"{c['pano_id'][:10]}: {len(proposed)} proposed, {len(res.valid_edges)} kept, "
        f"{len(res.rejected_edges)} rejected {[e.reason for e in res.rejected_edges]}"
    )
n_kept = sum(len(r.valid_edges) for r in results)
print(f"accepted_edges={n_kept}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

EDGE_COLORS = {  # BGR for cv2; the legend converts to RGB
    "RIDGE": (0, 0, 255),
    "EAVE": (0, 200, 0),
    "HIP": (255, 0, 0),
    "VALLEY": (255, 0, 255),
    "RAKE": (0, 200, 255),
}
REJECTED_COLOR = (160, 160, 160)


def draw(img, res):
    out = img.copy()
    for e in res.rejected_edges:
        pts = np.array(e.points, float)
        for a, b in zip(pts[:-1], pts[1:], strict=True):  # dashed grey
            n = max(2, int(np.linalg.norm(b - a) // 12))
            for i in range(0, n - 1, 2):
                p = a + (b - a) * i / n
                q = a + (b - a) * (i + 1) / n
                cv2.line(out, tuple(p.astype(int)), tuple(q.astype(int)), REJECTED_COLOR, 2)
    for e in res.valid_edges:
        cv2.polylines(out, [np.array(e.points, np.int32)], False, EDGE_COLORS[e.edge_type], 4)
    return out


fig, axes = plt.subplots(1, len(chosen), figsize=(7 * len(chosen), 6))
for ax, c, res in zip(np.atleast_1d(axes), chosen, results, strict=True):
    ax.imshow(draw(c["image"], res)[..., ::-1])
    attribution.add_to_axes(ax)
    ax.set_title(
        f"{c['dist_m']:.0f} m | kept {len(res.valid_edges)}, rejected {len(res.rejected_edges)}"
        f"\ngrad_sup {res.mean_gradient_support:.2f} | "
        f"random accept {res.random_acceptance:.3f}"
    )
    ax.axis("off")
legend = [
    Line2D([0], [0], color=np.array(bgr[::-1]) / 255, lw=4, label=t)
    for t, bgr in EDGE_COLORS.items()
]
legend.append(
    Line2D([0], [0], color=np.array(REJECTED_COLOR) / 255, lw=2, ls="--", label="rejected")
)
fig.legend(handles=legend, loc="lower center", ncol=len(legend))
plt.show()